In [ ]:
!pip install "gymnasium[box2d]"
!pip install swig

# **IMPORTING LIBRARIES**

In [ ]:
import torch
import random
import os
import torch.nn as nn
import numpy as np
import time
from collections import deque
import wandb
from tqdm import tqdm
from torch.amp import autocast, GradScaler

import gymnasium as gym
import ale_py
from gymnasium.wrappers import RecordVideo

In [ ]:
def set_seed(seed):
    import random, numpy as np, torch

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


In [ ]:
def create_environment(cfgs, eval = False):
  env = gym.make( id=cfgs.id , render_mode="rgb_array")
  return env

# **WANDB RUN**

In [ ]:
def wandb_runs(cfg):

  wandb.login(key = xxx)
  run = wandb.init(
    entity=xxx,
    #project="Lunar-Lander",
    project="CARTPOLE",
    name = f"VANILLA-PPO-{seed}-{cfg.id}",
    config=vars(cfg),
  )

  return run

# **CONFIGURATIONS**

In [ ]:
from dataclasses import dataclass

@dataclass
class configuration:
  id = "CartPole-v1" #"LunarLander-v3"
  max_training_step = 500_000 #1_000_000
  rollout_steps = 1400
  eval_steps = 10_000 #20_000
  global_steps = 0
  eval_loops = 3
  batch_size = 96
  d_model = 64
  ppo_r_clamp = 0.2
  critic_lr = 5e-4
  actor_lr = 3e-4
  record_video = 200_000
  wandblog_step = 100
  device = "cuda" if torch.cuda.is_available() else "cpu"

cfg = configuration()

**SyncVectorEnv so that we can run the n-environments parrallelly and utilize the GPUs because single environment is wayy poor**

In [ ]:
envs = create_environment(cfg)

**Checking environment is working or not:)**

# **Actor and Critic Netowrk**

In [ ]:
class Actor(nn.Module):

  def __init__(self, input_dim, action_dim):
    super().__init__()
    self.sequential = nn.Sequential(
        nn.Linear(input_dim, cfg.d_model),
        nn.ReLU(),
        nn.Linear(cfg.d_model, 2*cfg.d_model),
        nn.ReLU(),
        nn.Linear(2*cfg.d_model, cfg.d_model),
        nn.ReLU(),
        nn.Linear(cfg.d_model, action_dim),
    )

  def forward(self, x):
    x = self.sequential(x)
    return x

  def get_log_probs_action(self, x, actions = None):
    logits = self(x)

    dist = torch.distributions.Categorical(logits=logits)

    if actions == None:
      action = dist.sample()

    else:
      action = actions

    log_prob = dist.log_prob(action)
    entropy = dist.entropy().mean()

    return action, log_prob, entropy

In [ ]:
class Critic(nn.Module):

  def __init__(self, input_dim):
    super().__init__()

    self.sequential = nn.Sequential(
        nn.Linear(input_dim, cfg.d_model),
        nn.ReLU(),
        nn.Linear(cfg.d_model, 2*cfg.d_model),
        nn.ReLU(),
        nn.Linear(2*cfg.d_model, cfg.d_model),
        nn.ReLU(),
        nn.Linear(cfg.d_model, 1)
    )

  def forward(self, x):
    x = self.sequential(x)
    return x

In [ ]:
actornet = Actor(envs.observation_space.shape[0], envs.action_space.n).to(cfg.device)  #type: ignore
criticnet = Critic(envs.observation_space.shape[0]).to(cfg.device)  #type: ignore

In [ ]:
print(f'''Parameters:
==============================
actor-network     : {sum(p.numel() for p in actornet.parameters())/1e3} k
critic-network(s) : {sum(p.numel() for p in criticnet.parameters())/ 1e3} k
==============================
      ''')

**Evaluation Loop**

In [ ]:
def evaluation(actornet, record_video = False):

  eval_env = gym.make(id = cfg.id, render_mode = 'rgb_array')
  if record_video:
    video_dir = f"videos/{int(time.time())}"
    eval_env = RecordVideo(eval_env,  video_folder=video_dir, episode_trigger=lambda ep: True)

  net_reward = 0
  net_step = 0

  with torch.no_grad():

    for _ in range(cfg.eval_loops):

      done = False

      episodic_reward = 0
      episodic_step = 0
      state = eval_env.reset()[0]

      while not done:

        stateT = torch.tensor(state, dtype=torch.float32, device=cfg.device)

        with autocast(device_type=cfg.device, dtype=torch.float16):
          action = actornet(stateT).argmax().item()
        nxt_state, reward, terminated, truncated, _ = eval_env.step(action)
        done = terminated or truncated
        state = nxt_state

        episodic_reward += float(reward)
        episodic_step += 1

      net_reward += episodic_reward
      net_step  += episodic_step

  net_reward = net_reward / cfg.eval_loops
  net_step = net_step / cfg.eval_loops

  eval_env.close()

  return net_reward, net_step

In [ ]:
evaluation(actornet, True)

**To sample the batches**

**W&B RUNS TO LOG THE METRICS**

In [ ]:
gamma = 0.99
lambda_ = 0.98
entropy_beta = 0.01

In [ ]:
all_seeds = [0, 1, 2,  3, 4]

In [ ]:
for seed in all_seeds:

  runs = wandb_runs(cfg)

  envs = create_environment(cfg)

  set_seed(seed)

  cfg.global_steps = 0

  actornet = Actor(envs.observation_space.shape[0], envs.action_space.n).to(cfg.device)
  criticnet = Critic(envs.observation_space.shape[0]).to(cfg.device)

  actor_optimizer = torch.optim.AdamW(actornet.parameters(), lr = cfg.actor_lr)
  critic_optimizer = torch.optim.AdamW(criticnet.parameters(), lr = cfg.critic_lr)

  scaler_actor = GradScaler(enabled=(cfg.device == "cuda"))
  scaler_critic = GradScaler(enabled=(cfg.device == "cuda"))

  state = envs.reset(seed=seed)[0]

  rollout = 1

  while cfg.max_training_step > cfg.global_steps:

    states = []
    actions = []
    next_states = []
    rewards = []
    log_probs = []
    dones = []

    training_reward = 0
    training_step = 0

    rollout_step = 0

    stateT = torch.tensor(state, dtype=torch.float32, device=cfg.device)

    while rollout_step < cfg.rollout_steps and cfg.max_training_step > cfg.global_steps:

      with torch.no_grad():
        with autocast(device_type=cfg.device, dtype=torch.float16, enabled=(cfg.device == "cuda")):
          action, logprob, _ = actornet.get_log_probs_action(stateT, None)

      next_state, reward, terminated, truncated, _ = envs.step(action=action.item())

      done = terminated or truncated

      next_stateT = torch.tensor(next_state, dtype=torch.float32, device=cfg.device)
      rewardT = torch.tensor(reward, dtype=torch.float32, device=cfg.device)
      doneT = torch.tensor(done, dtype=torch.float32, device=cfg.device)

      states.append(stateT)
      actions.append(action)
      next_states.append(next_stateT)
      log_probs.append(logprob)
      rewards.append(rewardT)
      dones.append(doneT)

      stateT = next_stateT

      training_reward += float(reward)
      training_step += 1
      rollout_step += 1

      if cfg.global_steps % cfg.eval_steps == 0:

        net_reward, net_step = evaluation(actornet, False)

        runs.log({
            "eval-reward": net_reward,
            "eval-length": net_step
        }, step=cfg.global_steps)

      cfg.global_steps += 1

      if done:
        state = envs.reset()[0]
        stateT = torch.tensor(state, dtype=torch.float32, device=cfg.device)
      else:
        state = next_state

    all_states = torch.stack(states)
    all_actions = torch.stack(actions).view(-1,1)
    all_next_states = torch.stack(next_states)
    all_rewards = torch.stack(rewards).view(-1,1)
    all_dones = torch.stack(dones).view(-1,1)
    old_log_probs = torch.stack(log_probs).view(-1,1)

    with torch.no_grad():

      values = criticnet(all_states).view(-1,1)

      if all_dones[-1]:
        next_value = torch.zeros((1,1), device=cfg.device)
      else:
        next_value = criticnet(all_next_states[-1]).view(1,1)

    T = all_rewards.size(0)

    advantages = torch.zeros_like(all_rewards)

    gae = 0

    for t in reversed(range(T)):

      if t == T - 1:
        next_val = next_value
      else:
        next_val = values[t+1]

      delta = all_rewards[t] + gamma * (1 - all_dones[t]) * next_val - values[t]

      gae = delta + gamma * lambda_ * (1 - all_dones[t]) * gae

      advantages[t] = gae

    returns = advantages + values

    advantages = (advantages - advantages.mean()) / (advantages.std(unbiased=False) + 1e-8)

    value_bias = (returns - values).mean().item()

    explained_var = 1 - ((returns - values).var() / (returns.var() + 1e-8))

    log_actor_loss = 0
    log_policy_loss = 0
    log_critic_loss = 0
    log_entropy = 0
    step = 0

    shuffled_ids = torch.randperm(all_states.size(0), device=cfg.device)

    batch_size = min(cfg.batch_size, all_states.size(0))

    log_actor_grad_norm = 0
    log_critic_grad_norm = 0

    kl_divergence = 0

    for i in range(0, all_actions.size(0), batch_size):

      batch_ids = shuffled_ids[i:i+batch_size]

      mb_states = all_states[batch_ids]
      mb_rewards = all_rewards[batch_ids].squeeze(-1)
      mb_actions = all_actions[batch_ids].squeeze(-1)
      mb_advantages = advantages[batch_ids].squeeze(-1)
      mb_returns = returns[batch_ids].squeeze(-1)
      mb_old_log_probs = old_log_probs[batch_ids].squeeze(-1)

      with autocast(device_type=cfg.device, dtype=torch.float16, enabled=(cfg.device == "cuda")):

        _, mb_new_log_probs, entropy = actornet.get_log_probs_action(mb_states, mb_actions)

        ratio = torch.exp(mb_new_log_probs - mb_old_log_probs)

        policy_loss_ = -torch.min(
            ratio * mb_advantages,
            torch.clamp(ratio, 1-cfg.ppo_r_clamp, 1+cfg.ppo_r_clamp) * mb_advantages
        ).mean()

        policy_loss = policy_loss_ - entropy_beta * entropy

        critic_loss = torch.nn.functional.mse_loss(
            criticnet(mb_states).squeeze(-1),
            mb_returns
        )

      kl_divergence += (mb_old_log_probs - mb_new_log_probs.float()).mean().item()

      log_actor_loss += policy_loss.item()
      log_critic_loss += critic_loss.item()
      log_entropy += entropy.item()
      log_policy_loss += policy_loss_.item()
      step += 1

      actor_optimizer.zero_grad()

      if cfg.device == "cuda":
          scaler_actor.scale(policy_loss).backward()
          scaler_actor.unscale_(actor_optimizer)
          actor_grad_norm = torch.nn.utils.clip_grad_norm_(actornet.parameters(), max_norm=1)
          scaler_actor.step(actor_optimizer)
          scaler_actor.update()
      else:
          policy_loss.backward()
          actor_grad_norm = torch.nn.utils.clip_grad_norm_(actornet.parameters(), max_norm=1)
          actor_optimizer.step()

      critic_optimizer.zero_grad()

      if cfg.device == "cuda":
          scaler_critic.scale(critic_loss).backward()
          scaler_critic.unscale_(critic_optimizer)
          critic_grad_norm = torch.nn.utils.clip_grad_norm_(criticnet.parameters(), max_norm=1)
          scaler_critic.step(critic_optimizer)
          scaler_critic.update()
      else:
          critic_loss.backward()
          critic_grad_norm = torch.nn.utils.clip_grad_norm_(criticnet.parameters(), max_norm=1)
          critic_optimizer.step()

      log_actor_grad_norm += actor_grad_norm.item()
      log_critic_grad_norm += critic_grad_norm.item()

    runs.log({
        "env-number": rollout,
        "advantage-mean": advantages.mean().item(),
        "advantage-var": advantages.var(unbiased=False).item(),
        "actor-gradient-norm": log_actor_grad_norm/step,
        "critic-gradient-norm": log_critic_grad_norm/step,
        "policy-loss": log_policy_loss/step,
        "actor-loss": log_actor_loss/step,
        "critic-loss": log_critic_loss/step,
        "explained-variance": explained_var,
        "entropy": log_entropy/step,
        "kl-divergence": kl_divergence/step,
        "value-bias": value_bias,
        "training-reward": training_reward,
        "global-steps": cfg.global_steps
    }, step=cfg.global_steps)

    rollout += 1

In [ ]:
scaler_actor.unscale_(actor_optimizer)

total_norm = torch.norm(torch.stack([
    p.grad.norm() for p in actornet.parameters()
    if p.grad is not None
]))

print("true grad norm:", total_norm.item())

In [ ]:
print("adv max:", mb_advantages.abs().max().item())
print("ratio max:", ratio.max().item())
print("logprob:", mb_new_log_probs.abs().max().item())

In [ ]:
kl_divergence

In [ ]:
actor_norm = torch.nn.utils.clip_grad_norm_(actornet.parameters(), 1e9)
critic_norm = torch.nn.utils.clip_grad_norm_(criticnet.parameters(), 1e9)

print("actor:", actor_norm, "critic:", critic_norm)

# **SAVE MODEL-WEIGHTS**

In [ ]:
torch.save(actornet.state_dict(), "policy-weights.pt")

In [ ]:
actor_optimizer.param_groups[0]['lr']